In [1]:
import numpy as np
import json


In [2]:
import requests

In [3]:
from sentence_transformers import SentenceTransformer

c:\Users\Alice\anaconda3\envs\yolo\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# 导入所需库


# 2. 加载嵌入模型
#embedding_model = SentenceTransformer('/home/youlika/models/bert-base-uncased')
embedding_model = SentenceTransformer(
    r"C:\Users\Alice\.cache\huggingface\hub\models--google-bert--bert-base-uncased\snapshots\86b5e0934494bd15c9632b12f734a8a67f723594"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2624.36it/s]
[transformers] BertModel LOAD REPORT from: C:\Users\Alice\.cache\huggingface\hub\models--google-bert--bert-base-uncased\snapshots\86b5e0934494bd15c9632b12f734a8a67f723594
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:



# 3. 准备示例文档
documents = [
    "尤里卡的大模型应用开发课程需要的前置知识是Python。",
    "具身智能导学课程的前置知识是多模态大模型。",
    "尤里卡是中南大学的博士研究生，硕士毕业于中国科学院大学，同时是一名创业者。",
    "尤里卡在读博之前，曾在工业界工作了6年，其中1年在创业。目前一边读博，一边做AI前沿技术的培训",
    "目前尤里卡主要运营的社交平台包括：bilibili、小红书。"
]

# 4. 为文档生成嵌入
document_embeddings = embedding_model.encode(documents, convert_to_tensor=False)
document_embeddings = np.array(document_embeddings)

# 1. 初始化设置
api_key = "your_deepseek_api_key_here"  # 替换为你的API密钥
model_name = "deepseek-chat"
api_url = "https://api.deepseek.com/v1/chat/completions"

# 5. 用户查询
user_query = "尤里卡在哪儿上大学？"

# 6. 为查询生成嵌入
query_embedding = embedding_model.encode(user_query, convert_to_tensor=False)

# 7. 计算相似度
similarities = np.dot(document_embeddings, query_embedding) / (
    np.linalg.norm(document_embeddings, axis=1) * np.linalg.norm(query_embedding)
)






In [13]:
document_embeddings

array([[ 0.05830933,  0.4071372 ,  0.22232205, ..., -0.6514841 ,
         0.7333131 , -0.27336442],
       [ 0.10242483,  0.17852625,  0.1756093 , ..., -0.52467245,
         0.589721  , -0.46089196],
       [ 0.02857606,  0.19503392,  0.30644754, ..., -0.4485348 ,
         0.35223588, -0.45638612],
       [-0.00719952,  0.4273535 ,  0.23163877, ..., -0.34224007,
         0.4999361 , -0.4422358 ],
       [ 0.00641133,  0.2173081 ,  0.14217345, ..., -0.39322704,
         0.27947602, -0.23249152]], shape=(5, 768), dtype=float32)

In [14]:
query_embedding

array([-9.98485982e-02, -7.94973597e-02, -9.87760052e-02, -5.38273871e-01,
       -3.47269207e-01, -8.97447690e-02,  4.83012319e-01, -6.01399131e-02,
       -6.40838370e-02, -7.09607527e-02,  1.56355828e-01, -4.87976037e-02,
        2.80481607e-01, -1.41457006e-01, -1.62838742e-01, -1.50309309e-01,
        1.41566858e-01, -2.71843195e-01, -1.49366111e-02,  8.37213337e-01,
        1.41452983e-01,  2.50283778e-01, -3.42298180e-01,  2.62561411e-01,
        2.41140842e-01,  3.81000549e-01,  3.12458724e-01, -2.01796591e-01,
       -3.24423134e-01, -2.89551407e-01,  7.39417017e-01, -1.99803591e-01,
       -2.08959088e-01,  2.87299544e-01,  4.28125530e-01, -2.54807949e-01,
        4.13471274e-02,  2.40186956e-02, -5.98346107e-02,  2.47729778e-01,
       -4.34815049e-01, -5.57192266e-01,  2.85441548e-01, -2.07247958e-01,
       -6.10269904e-01, -7.73321390e-01, -1.61927462e-01,  3.25669278e-03,
        1.30445793e-01,  1.92848042e-01, -6.19822145e-01,  4.89870124e-02,
       -1.41968131e-01, -

In [15]:
similarities

array([0.8862287 , 0.92478687, 0.88548064, 0.8696389 , 0.8959803 ],
      dtype=float32)

In [16]:
# 8. 获取最相关文档
top_k = 2
most_similar_indices = np.argsort(similarities)[-top_k:][::-1]
relevant_docs = [documents[i] for i in most_similar_indices]



In [17]:
relevant_docs

['具身智能导学课程的前置知识是多模态大模型。', '目前尤里卡主要运营的社交平台包括：bilibili、小红书。']

In [18]:
user_query

'尤里卡在哪儿上大学？'

In [ ]:
# 9. 构建提示
prompt = f"""基于以下上下文信息回答问题。如果上下文不足以回答问题，请说明。

上下文:
{''.join([f'- {doc}\n' for doc in relevant_docs])}

问题: {user_query}

回答:"""
print("输入给大模型的prompt是：", prompt)

response = client.chat.completions.create(
    model=model_name,
    messages=[
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": prompt},
    ],
    stream=False
)

print(response.choices[0].message.content)

In [ ]:

# 10. 调用DeepSeek API
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

payload = {
    "model": model_name,
    "messages": [
        {"role": "system", "content": "你是一个有帮助的AI助手，基于提供的上下文回答问题。"},
        {"role": "user", "content": prompt}
    ],
    "temperature": 0.7,
    "max_tokens": 1000
}


In [ ]:
# 11. 发送请求并获取响应
response = requests.post(api_url, headers=headers, data=json.dumps(payload))
response.raise_for_status()

# 12. 输出结果
generated_answer = response.json()["choices"][0]["message"]["content"]
print("问题:", user_query)
print("检索到的相关文档:", relevant_docs)
print("生成的回答:", generated_answer)